### `wst-eccentricity`

This notebook walks through the full training/evaluation path of the
`wst-eccentricity` package step by step. It mirrors
[`examples/quickstart.py`](./quickstart.py) but is broken into small, runnable
cells you can read and modify interactively.

It has two modes:

* **Demo mode** (default): trains on small *synthetic* 4D tensors, so you can
  exercise the API without downloading any data.
  
* **Real-data mode**: point `DATA_DIR` at a prepared dataset
  (`waveforms/` + `parameters/`) and the notebook computes the wavelet
  scattering transform (WST), builds a labelled dataset, and trains on it.

The classifier is `SWT_CNN_1D_Binned`, which consumes the **native** scattering
tensor of shape `(N, D, C, T)` = (signals, detectors, scattering channels,
time) — the detector/channel/time axes are kept separate rather than
flattened.

### 1. Configuration

Set `DEMO = True` for the no-data synthetic run. To use a real dataset, set
`DEMO = False` and point `DATA_DIR` at a folder laid out as:

```
DATA_DIR/
    waveforms/    # <name>.hdf5 per signal
    parameters/   # params_*.txt per signal (must contain an 'eccentricity:' line)
```

In [ ]:
import torch

DEMO = True                 # set False to use a real dataset
DATA_DIR = "data"           # only used when DEMO = False

J, Q = 7, 2                 # wavelet scattering hyper-parameters
E_THR = 0.01                # eccentricity > E_THR  ->  positive class
EPOCHS = 20
TARGET_FPR = 0.1            # operating point: 10% false-positive rate
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

### 2. Load features

Either build a synthetic `(N, D, C, T)` tensor (demo) or run the real
waveforms → WST → labelled-dataset pipeline. In both cases we end up with:

* `X` — features of shape `(N, D, C, T)`
* `y` — binary labels of shape `(N,)`

In [ ]:
from wst_eccentricity import (
    build_dataset,
    load_hdf5_data,
    match_waveform_and_parameter_files,
    scatter_in_batches,
)

if DEMO:
    # Synthetic 4D scattering-like tensor with a mild class-dependent shift,
    # so the classifier has something learnable.
    torch.manual_seed(0)
    n, D, C, T = 400, 3, 8, 32
    y = torch.randint(0, 2, (n,))
    X = torch.randn(n, D, C, T) + y.view(-1, 1, 1, 1) * 0.5
    y = y.long()
else:
    # Real data: waveforms -> WST (cached) -> labelled dataset.
    # Waveform and parameter files are paired by their s<seed>_<index> tag,
    # so features and labels are guaranteed to stay aligned.
    import os

    wf_files, pf_files = match_waveform_and_parameter_files(
        f"{DATA_DIR}/waveforms", f"{DATA_DIR}/parameters"
    )
    gws, _ = load_hdf5_data(files=wf_files)
    scatter_dir = f"{DATA_DIR}/transform_coefficients"
    scatter_in_batches(
        gws, J, Q, out_dir=scatter_dir, device=DEVICE,
        names=[os.path.basename(f) for f in wf_files],
    )
    X, y, _params = build_dataset(
        params_dir=None,
        wst_dir=scatter_dir,
        J=J,
        Q=Q,
        e_thr=E_THR,
        param_files=pf_files,
    )

print("features X:", tuple(X.shape), "  labels y:", tuple(y.shape))
print("positives:", int(y.sum()), "/", len(y))

## 3. Standardize and split

`standardize` normalizes each feature to zero mean / unit variance across the
batch axis (it works on the native 4D shape — no flattening). We then split
into train / validation / test sets.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset, random_split
from wst_eccentricity import standardize

X = standardize(X)

ds = TensorDataset(X, y)
n_val = max(1, int(0.2 * len(ds)))
n_test = max(1, int(0.2 * len(ds)))
n_train = len(ds) - n_val - n_test
train_ds, val_ds, test_ds = random_split(
    ds, [n_train, n_val, n_test], generator=torch.Generator().manual_seed(0)
)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128)
test_loader = DataLoader(test_ds, batch_size=128)
print(f"train/val/test = {n_train}/{n_val}/{n_test}")

## 4. Build the model

`SWT_CNN_1D_Binned` infers nothing from the data on its own, so we read the
detector count `D` and scattering-channel count `C` straight from `X`
(`X.shape = (N, D, C, T)`). Other hyper-parameters (`time_bins`,
`cnn_channels`, `kernel_sizes`, `dropout_rate`) have sensible defaults.

In [ ]:
from wst_eccentricity import SWT_CNN_1D_Binned

model = SWT_CNN_1D_Binned(in_channels=X.shape[2], num_detectors=X.shape[1])
model.info()  # prints the model name and trainable-parameter count

## 5. Train

`train_binary` runs a plain-PyTorch training loop with early stopping on the
validation AUC, and restores the best weights before returning.

In [ ]:
from wst_eccentricity import train_binary

model, history, best_val_auc = train_binary(
    model, train_loader, val_loader, max_epochs=EPOCHS, device=DEVICE
)
print(f"Best validation AUC: {best_val_auc:.3f}")

## 6. Evaluate

We pick the decision threshold on the **validation** set (at the target FPR),
then report AUC, average precision, and the achieved FPR/TPR on the **test**
set.

In [ ]:
from sklearn.metrics import roc_curve
from wst_eccentricity import (
    auc_ap,
    collect_probs_targets,
    fpr_tpr_from_counts,
    threshold_for_target_fpr,
)
from wst_eccentricity.metrics import confusion_counts

# Threshold chosen on validation.
val_t, val_p = collect_probs_targets(model, val_loader, DEVICE)
fpr, tpr, thr = roc_curve(val_t, val_p, pos_label=1)
tau, _, _ = threshold_for_target_fpr(fpr, tpr, thr, TARGET_FPR)

# Evaluate on test.
test_t, test_p = collect_probs_targets(model, test_loader, DEVICE)
test_auc, test_ap = auc_ap(test_t, test_p)
tn, fp, fn, tp = confusion_counts(test_t, test_p, tau)
fpr_at, tpr_at = fpr_tpr_from_counts(tn, fp, fn, tp)

print(f"Test AUC:               {test_auc:.3f}")
print(f"Test average precision: {test_ap:.3f}")
print(f"At target FPR={TARGET_FPR:.2f} (tau={tau:.3f}): "
      f"FPR={fpr_at:.3f}, TPR={tpr_at:.3f}")

## 7. One-call pipeline

For real data you can skip the manual steps above and run the whole thing in a
single call. This computes the WST (cached), builds the dataset, trains the
chosen classifier, and returns the metrics dict.

```python
from wst_eccentricity import run_pipeline

results = run_pipeline("data", J=7, Q=2, classifier="cnn1d", target_fpr=0.1)
print(results["auc"], results["tpr"])
```